# 02 · Extract satellite signals (Google Earth Engine)

**Purpose:** measure, for every village, how much its night-time lights and built-up area changed over the growth window.

| | |
|---|---|
| **Inputs** | Earth Engine assets `villages_06_shp` and `villages_india_shp` (uploaded at the end of notebook 01) |
| **Outputs** | CSVs exported to Google Drive folder `village_growth`, then downloaded to `data/processed/`: `lights_built_06.csv`, `buildings_06.csv` (pilot) and `lights_built_india_<state>.csv` × 33 (national, Stage 1) |
| **Next step** | `03_score_rank` (which also runs the Stage 2 buildings export for the shortlist) |
| **Requirements** | An Earth Engine account + Cloud project; `earthengine authenticate` done once |

**Pipeline position:** 01 Prepare → **02 Extract** → 03 Score & rank → 04 Visualise

>  **The exports are switched off by default** (`RUN_*_EXPORT = False`) because the CSVs already exist in `data/processed/` and re-running them uses Earth Engine compute quota. Set a flag to `True` to re-extract.

In [16]:
import ee
from collections import Counter
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
PROC = ROOT / "data" / "processed"

# ---- Earth Engine settings ----
PROJECT_ID = "stately-sentry-472119-i4"                 # change to your own Cloud project to reproduce
ASSET_PILOT = f"projects/{PROJECT_ID}/assets/villages_06_shp"
ASSET_INDIA = f"projects/{PROJECT_ID}/assets/villages_india_shp"
DRIVE_FOLDER = "village_growth"                          # Google Drive folder that receives the CSVs

# ---- Growth window ----
START, END = 2020, 2025          # night lights + Dynamic World (5 full calendar years)
OB_START, OB_END = 2018, 2023    # Open Buildings Temporal only goes up to 2023

# ---- Switches (see warning above) ----
RUN_PILOT_EXPORT = False
RUN_NATIONAL_EXPORT = False

INDIA_BBOX = [68, 6, 98, 37]     # lon/lat bounding box of India, used to pre-filter image collections

ee.Initialize(project=PROJECT_ID)
print("Earth Engine ready")

Earth Engine ready


>  **Why 2020 → 2025:** VIIRS data runs to Sept 2026, so 2025 is the latest full year. The brief asks for 5 years. 2020 = COVID year; 95% of villages brightened nationally.

## The three growth signals

| Signal | Dataset | What we compute per village | Why |
|---|---|---|---|
| **Night lights** | NOAA VIIRS DNB monthly (`VCMSLCFG`), ~500 m | Mean of the yearly *median* radiance | Widely used proxy for economic activity; the median ignores one-off months (fires, clouds, glare) |
| **Built-up share** | Google/WRI Dynamic World, 10 m | Share of the village whose most common class in the year is *built* (class 6) | Captures new construction and land conversion |
| **Building count** | Google Open Buildings 2.5D Temporal (rasters at 0.5 m, effective 4 m) | Sum of `building_fractional_count` | Independent confirmation of physical growth (used in Stage 2) |

In [17]:
def night_lights(year):
    # Median of the 12 monthly composites, negatives (sensor noise) clipped to 0
    return (ee.ImageCollection("NOAA/VIIRS/DNB/MONTHLY_V1/VCMSLCFG")
            .filterDate(f"{year}-01-01", f"{year + 1}-01-01")
            .select("avg_rad").median().max(0)
            .rename(f"ntl_{year}"))


def built_up(year, region):
    # 1 where the most frequent Dynamic World label in the year is "built" (6), else 0.
    # Averaging this over a village gives the built-up SHARE of the village.
    return (ee.ImageCollection("GOOGLE/DYNAMICWORLD/V1")
            .filterDate(f"{year}-01-01", f"{year + 1}-01-01")
            .filterBounds(region)
            .select("label").mode().eq(6)
            .rename(f"built_{year}"))


def buildings(year, region):
    # Open Buildings Temporal: one image per tile per year -> mosaic the tiles of that year.
    # Summing building_fractional_count over a village estimates its number of buildings.
    # NOTE: the rasters are stored at 0.5 m but we sum at 4 m, so each value is an 8x8 average;
    #       the counts are multiplied by 64 in notebook 03 to get real building counts.
    return (ee.ImageCollection("GOOGLE/Research/open-buildings-temporal/v1")
            .filterBounds(region)
            .filterDate(f"{year}-01-01", f"{year + 1}-01-01")
            .select("building_fractional_count").mosaic()
            .rename(f"bldg_{year}"))


def export_csv(fc, name):
    # Export a FeatureCollection to Google Drive as CSV, without geometries (keeps files small)
    task = ee.batch.Export.table.toDrive(
        collection=fc.select([".*"], None, False),
        description=name, folder=DRIVE_FOLDER, fileNamePrefix=name, fileFormat="CSV")
    task.start()
    return task


def task_summary(tasks):
    # Quick progress check: READY = queued, RUNNING, COMPLETED, FAILED
    return Counter(t.status()["state"] for t in tasks)

## Part A · Pilot (Haryana)

All three signals at full resolution (10 m for lights/built-up, 4 m for buildings). Used to check the method before scaling up – results are analysed in the appendix of notebook 03.

In [18]:
pilot = ee.FeatureCollection(ASSET_PILOT)
pilot_region = pilot.geometry().bounds()
print("Pilot villages in Earth Engine:", pilot.size().getInfo())

pilot_tasks = []
if RUN_PILOT_EXPORT:
    lights_built = ee.Image.cat([night_lights(START), night_lights(END),
                                 built_up(START, pilot_region), built_up(END, pilot_region)])
    stats_a = lights_built.reduceRegions(collection=pilot, reducer=ee.Reducer.mean(),
                                         scale=10, tileScale=4)

    bldg = ee.Image.cat([buildings(OB_START, pilot_region), buildings(OB_END, pilot_region)])
    stats_b = bldg.reduceRegions(collection=pilot, reducer=ee.Reducer.sum(),
                                 scale=4, tileScale=8)

    pilot_tasks = [export_csv(stats_a, "lights_built_06"), export_csv(stats_b, "buildings_06")]
    print("Pilot exports started")
else:
    print("Pilot export skipped (RUN_PILOT_EXPORT = False)")

Pilot villages in Earth Engine: 6681
Pilot export skipped (RUN_PILOT_EXPORT = False)


## Part B · National Stage 1 (lights + built-up, all India)

One export task per state so they run in parallel and a failure only affects one state.
Run at **30 m** instead of 10 m to stay within the Earth Engine noncommercial compute quota – built-up shares are slightly smoother but rankings are comparable.

> Open Buildings is **not** run nationally (4 m over 590k villages is too expensive); it is only run on the shortlist in notebook 03 (Stage 2).

In [19]:
india = ee.FeatureCollection(ASSET_INDIA)
india_region = ee.Geometry.Rectangle(INDIA_BBOX)
states = india.aggregate_array("state").distinct().sort().getInfo()
print(len(states), "states:", states)

national_tasks = []
if RUN_NATIONAL_EXPORT:
    lights_built = ee.Image.cat([night_lights(START), night_lights(END),
                                 built_up(START, india_region), built_up(END, india_region)])
    for st in states:
        fc = india.filter(ee.Filter.eq("state", st))
        stats = lights_built.reduceRegions(collection=fc, reducer=ee.Reducer.mean(),
                                           scale=30, tileScale=8)
        national_tasks.append(export_csv(stats, f"lights_built_india_{st}"))
    print(len(national_tasks), "national exports started")
else:
    print("National export skipped (RUN_NATIONAL_EXPORT = False)")

33 states: ['01', '02', '03', '05', '06', '08', '09', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '30', '31', '32', '33', '34', '35']
National export skipped (RUN_NATIONAL_EXPORT = False)


In [20]:
# Re-run every few minutes while exports are running (or watch the Tasks tab in the Code Editor)
if pilot_tasks or national_tasks:
    print(task_summary(pilot_tasks + national_tasks))

## Manual step · Downloading the CSVs

When all tasks show **COMPLETED**: opened Google Drive → `village_growth`, downloaded the CSVs and placed them directly in `data/processed/`. The check below confirms everything is in place.

In [21]:
national_files = sorted(PROC.glob("lights_built_india_*.csv"))
print("Pilot files present:", all((PROC / f).exists() for f in ["lights_built_06.csv", "buildings_06.csv"]))
print(f"National state files: {len(national_files)} (expected {len(states)})")

Pilot files present: True
National state files: 33 (expected 33)
